# Quantizing Language Models
Quantization stores weights (and optionally activations) in fewer bits - e.g. float32 (4 bytes) -> int8 (1 byte).
It does **not** change the number of parameters; it changes how many bytes each one takes.
So measure it with **bytes on disk / in memory**, plus the two things that can get worse or better:
**quality** (perplexity) and **speed** (latency).

Part 1 quantizes a small transformer built from scratch; Part 2 quantizes a real pre-trained LLM (OPT-125m).

Requires `pip install torch torchao transformers`. Runs on CPU.

In [1]:
import io
import copy
import math
import time
import warnings
import pandas as pd
import torch
import torch.nn as nn
from torchao.quantization import quantize_, Int8WeightOnlyConfig

torch.manual_seed(0)
device = torch.device("cpu")  # the int8 kernels compared here target CPU

def model_size_mb(model):
    """Serialized size of the weights - the real storage cost."""
    buffer = io.BytesIO()
    torch.save(model.state_dict(), buffer)
    return buffer.tell() / 1e6

def time_forward(model, inputs, reps=10):
    with torch.inference_mode():
        model(inputs)  # warm-up
        start = time.perf_counter()
        for _ in range(reps):
            model(inputs)
    return (time.perf_counter() - start) / reps * 1e3  # ms

# Part 1: A small transformer language model

In [2]:
# 1. Define a smaller, simpler Transformer block
class SmallTransformerBlock(nn.Module):
    def __init__(self, hidden_size, num_heads, intermediate_size):
        super().__init__()
        self.attention = nn.MultiheadAttention(hidden_size, num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(hidden_size)
        self.feed_forward = nn.Sequential(
            nn.Linear(hidden_size, intermediate_size),  # Smaller intermediate layer
            nn.ReLU(),
            nn.Linear(intermediate_size, hidden_size)
        )
        self.norm2 = nn.LayerNorm(hidden_size)

    def forward(self, x):
        attn_output, _ = self.attention(x, x, x)
        x = self.norm1(x + attn_output)
        x = self.norm2(x + self.feed_forward(x))
        return x

# 2. Create the smaller LLM
class SmallLLM(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers, num_heads, intermediate_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.layers = nn.ModuleList([
            SmallTransformerBlock(hidden_size, num_heads, intermediate_size)
            for _ in range(num_layers)
        ])
        self.lm_head = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        for layer in self.layers:
            x = layer(x)
        return self.lm_head(x)

# 3. Instantiate the model with smaller parameters
vocab_size = 10000  # Example vocabulary size
hidden_size = 128   # Reduced hidden size
num_layers = 4      # Fewer layers
num_heads = 4       # Fewer attention heads
intermediate_size = 256  # Smaller intermediate size in feed-forward layer

model = SmallLLM(vocab_size, hidden_size, num_layers, num_heads, intermediate_size).eval()

In [3]:
# 4. Post-Training Quantization: int8 weights for every nn.Linear (the feed-forward layers and lm_head).
# nn.MultiheadAttention and nn.Embedding are not nn.Linear modules, so they stay in float32.
quantized_model = copy.deepcopy(model)
quantize_(quantized_model, Int8WeightOnlyConfig())

input_ids = torch.randint(0, vocab_size, (1, 64))  # Batch size 1, sequence length 64
with torch.inference_mode():
    diff = (model(input_ids) - quantized_model(input_ids)).abs().max().item()

print(f"Parameters (original):  {sum(p.numel() for p in model.parameters()):,}")
print(f"Parameters (quantized): {sum(p.numel() for p in quantized_model.parameters()):,}  <- same count")
print(f"Size (original):  {model_size_mb(model):.2f} MB")
print(f"Size (quantized): {model_size_mb(quantized_model):.2f} MB")
print(f"Max change in output logits: {diff:.4f}")

Parameters (original):  3,099,920
Parameters (quantized): 3,099,920  <- same count
Size (original):  12.42 MB
Size (quantized): 7.85 MB
Max change in output logits: 0.0134


# Part 2: Quantizing a pre-trained LLM
`facebook/opt-125m` is used because its layers are standard `nn.Linear` modules.
(GPT-2 implements its layers as a custom `Conv1D` class, so code that targets `nn.Linear` silently skips them.)

Quality is measured with **perplexity** on held-out text (`Data/llm_eval_text.txt`). Perplexity must be averaged
over **tokens**, not over sentences or batches, so each batch's mean loss is weighted by its number of predicted tokens.

In [4]:
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length):
        self.encodings = [tokenizer(t, truncation=True, max_length=max_length, return_tensors="pt") for t in texts]

    def __len__(self):
        return len(self.encodings)

    def __getitem__(self, idx):
        input_ids = self.encodings[idx]["input_ids"].squeeze(0)
        return {"input_ids": input_ids, "labels": input_ids.clone()}

def collate(batch, pad_token_id):
    # Pad to the longest text in the batch; padded label positions are -100 so the loss ignores them
    longest = max(len(b["input_ids"]) for b in batch)
    input_ids = torch.full((len(batch), longest), pad_token_id)
    labels = torch.full((len(batch), longest), -100)
    attention_mask = torch.zeros((len(batch), longest), dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["input_ids"])
        input_ids[i, :n] = b["input_ids"]
        labels[i, :n] = b["labels"]
        attention_mask[i, :n] = 1
    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}

model_name = "facebook/opt-125m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:  # e.g. GPT-2 has no pad token
    tokenizer.pad_token = tokenizer.eos_token

texts = [p.strip() for p in open("Data/llm_eval_text.txt").read().split("\n\n") if p.strip()]
test_dataloader = DataLoader(TextDataset(texts, tokenizer, max_length=128), batch_size=4, shuffle=False,
                             collate_fn=lambda b: collate(b, tokenizer.pad_token_id))
print(f"{len(texts)} evaluation passages")

8 evaluation passages


In [5]:
def calculate_perplexity(model, dataloader):
    model.eval()
    total_nll, total_tokens = 0.0, 0
    with torch.inference_mode():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            loss = model(**batch).loss  # mean loss over the predicted (non -100) tokens in this batch
            # The model predicts token t+1 from tokens <= t, so the first label of each row is never predicted
            n_tokens = (batch["labels"][:, 1:] != -100).sum().item()
            total_nll += loss.float().item() * n_tokens
            total_tokens += n_tokens
    return math.exp(total_nll / total_tokens)

# Load explicitly in float32: recent transformers versions default to the checkpoint's dtype (float16 for OPT)
base_model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float32).to(device).eval()
print(f"Perplexity (float32): {calculate_perplexity(base_model, test_dataloader):.2f}")

Perplexity (float32): 31.66


## Compare quantization options
* **bfloat16** - every weight in 16 bits. Simple, but CPUs without native bf16 support run it *slower*.
* **int8 weight-only (torchao)** - `nn.Linear` weights stored as int8, de-quantized on the fly. `lm_head` is skipped
  because OPT shares (ties) it with the input embedding; quantizing it would store a second copy.
* **int8 weight-only + bf16 for the rest** - also shrinks the large embedding table.
* **int8 dynamic (legacy `torch.ao`)** - weights *and* activations in int8 using fast CPU kernels.
  Deprecated in favor of torchao, shown here because it illustrates the speed side of the trade-off.

In [6]:
skip_lm_head = lambda module, name: isinstance(module, nn.Linear) and name != "lm_head"

def make_variant(kind):
    m = copy.deepcopy(base_model)
    if kind == "bfloat16":
        m = m.to(torch.bfloat16)
    elif kind == "int8 weight-only":
        quantize_(m, Int8WeightOnlyConfig(), filter_fn=skip_lm_head)
    elif kind == "int8 weight-only + bf16":
        m = m.to(torch.bfloat16)
        quantize_(m, Int8WeightOnlyConfig(), filter_fn=skip_lm_head)
    elif kind == "int8 dynamic (torch.ao)":
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")  # deprecation warning
            m = torch.ao.quantization.quantize_dynamic(m, {nn.Linear}, dtype=torch.qint8)
    return m.eval()

sample = tokenizer(texts[0], return_tensors="pt").input_ids.to(device)
rows = []
for kind in ["float32", "bfloat16", "int8 weight-only", "int8 weight-only + bf16", "int8 dynamic (torch.ao)"]:
    m = make_variant(kind)
    rows.append({"variant": kind,
                 "size_MB": model_size_mb(m),
                 "perplexity": calculate_perplexity(m, test_dataloader),
                 "latency_ms": time_forward(m, sample)})
    del m

results = pd.DataFrame(rows).set_index("variant")
results["size_vs_fp32"] = results.size_MB / results.loc["float32", "size_MB"]
results["speedup_vs_fp32"] = results.loc["float32", "latency_ms"] / results.latency_ms
results.round(2)

,size_MB,perplexity,latency_ms,size_vs_fp32,speedup_vs_fp32
variant,,,,,
float32,501.03,31.66,50.15,1.00,1.00
bfloat16,250.55,31.77,208.90,0.50,0.24
int8 weight-only,246.69,31.53,62.42,0.49,0.80
int8 weight-only + bf16,166.08,31.66,222.88,0.33,0.23
int8 dynamic (torch.ao),284.89,36.48,30.85,0.57,1.63


In [7]:
# Eyeball check: greedy completions from the original and a quantized model
prompt = "The most important thing to remember when learning to cook is"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
for kind in ["float32", "int8 weight-only"]:
    m = make_variant(kind)
    with torch.inference_mode():
        out = m.generate(**inputs, max_new_tokens=25, do_sample=False)
    print(f"[{kind}] {tokenizer.decode(out[0], skip_special_tokens=True)}\n")

[float32] The most important thing to remember when learning to cook is that you don't have to cook everything.                



[int8 weight-only] The most important thing to remember when learning to cook is that you don't have to cook everything.                



## Takeaways
* Weight-only int8 roughly halves storage here with almost no perplexity change. It is not 4x smaller because the
  embedding table (~40M of the 125M parameters) is not an `nn.Linear` and stays in float32.
* The legacy dynamic-int8 path is the fastest on CPU but quantizes `lm_head` and activations too, which raises perplexity noticeably.
* Speed depends on hardware kernels: a format that saves memory can still be slower if the hardware lacks native support.
  Always measure on the deployment target.
* For larger models and GPUs, 4-bit methods (GPTQ, AWQ, bitsandbytes NF4, GGUF for llama.cpp) are the common next step.